# แสดงผลการ Preprocess (แบบเก่า 02_preprocess_hmc เทียบกับแบบใหม่ 03_Preprocess_multi5)

## Section 1 Setup


In [ ]:
!pip install mne -q
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/SleepStageClassificationProject"

OLD_DIR = f"{PROJECT_DIR}/dataset/preprocess_microvolt"
NEW_DIR = f"{PROJECT_DIR}/dataset/preprocess_microvolt_multi5"
FIG_DIR = f"{PROJECT_DIR}/dataset/preprocess_viz"

SUBJECTS = ["SN001", "SN004", "SN006", "SN002", "SN010"]

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path

STAGES = ["W", "N1", "N2", "N3", "R"]
N_CONSEC = 5
FS, EPOCH_LEN = 256, 30 * 256
Y_POS = {"W": 4, "R": 3, "N1": 2, "N2": 1, "N3": 0}

COLORS = {"W": "#FFD3A8", "N1": "#BCDDF7", "N2": "#9EC5F0", "N3": "#8C9EE0", "R": "#F7B2C9"}
C_OLD, C_NEW, C_AVAIL = "#F7A8B8", "#8FD3BE", "#D9D9E6"
LINE_DARK, LINE_LIGHT = "#4A4A5A", "#BDBDCC"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#FCFBFF",
    "axes.edgecolor": "#C9C9D6", "axes.labelcolor": LINE_DARK, "axes.titlecolor": LINE_DARK,
    "text.color": LINE_DARK, "xtick.color": "#6B6B7B", "ytick.color": "#6B6B7B",
    "grid.color": "#E6E6EF", "legend.edgecolor": "#E6E6EF",
})

os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name):
    path = f"{FIG_DIR}/{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("บันทึกแล้ว:", path)

## Section 2 กราฟที่ 1: epoch ก่อนและหลัง preprocess (ผู้ถูกทดสอบ 5 คน)

In [ ]:
from matplotlib.ticker import MultipleLocator

def find_run(labels, stage):
    best_first, best_len, count = None, 0, 0
    for i, lab in enumerate(labels):
        count = count + 1 if lab == stage else 0
        if count == N_CONSEC:
            return i - N_CONSEC + 1, N_CONSEC
        if count > best_len:
            best_len, best_first = count, i - count + 1
    return best_first, best_len

def find_all_runs(labels, stage):
    starts, i = [], 0
    while i + N_CONSEC <= len(labels):
        if all(labels[i + k] == stage for k in range(N_CONSEC)):
            starts.append(i); i += N_CONSEC
        else:
            i += 1
    return starts

def load_labels(data_dir, subject):
    import mne
    data_dir = Path(data_dir)
    raw = mne.io.read_raw_edf(data_dir / f"{subject}.edf", preload=False, verbose=False)
    n_epochs = raw.n_times // EPOCH_LEN
    ann = mne.read_annotations(data_dir / f"{subject}_sleepscoring.edf")
    labels = ["?"] * n_epochs
    for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
        stage = desc.split()[-1]
        if stage not in STAGES:
            continue
        for e in range(int(round(onset / 30)), int(round((onset + duration) / 30))):
            if e < n_epochs:
                labels[e] = stage
    return labels

def load_selected(base_dir, subject):
    for sp in ["train", "val", "test"]:
        f = Path(base_dir) / sp / f"{subject}.npz"
        if f.exists():
            with np.load(f, allow_pickle=True) as d:
                stages = [str(s) for s in d["stage"]]
                eps = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]
            return sp, stages, eps
    return None, [], []

def print_selected(name, stages, eps, max_rows=None):
    rows = sorted(
        [(int(ep.min()), int(ep.max()), s) for s, ep in zip(stages, eps) if len(ep)],
        key=lambda r: r[0],
    )
    print(f"\n{name}: {len(rows)} sets")
    if not rows:
        print("   (ไม่มีไฟล์ / ไม่มีชุดที่เลือก)")
        return

    first = {}
    for a, b, s in rows:
        first.setdefault(s, (a, b))
    print("   ชุดแรกของแต่ละ stage: " +
          " | ".join(f"{s}: {a}-{b}" for s, (a, b) in first.items()))

    shown = rows if max_rows is None else rows[:max_rows]
    for i, (a, b, s) in enumerate(shown, 1):
        print(f"   {i:>3}. {s:<2} epoch {a}-{b}  ({a * 30 / 60:.1f}-{(b + 1) * 30 / 60:.1f} นาที)")
    if max_rows is not None and len(rows) > max_rows:
        print(f"   ... อีก {len(rows) - max_rows} ชุด")

def style_hyp_axis(ax, hours):
    ax.set_yticks(list(Y_POS.values())); ax.set_yticklabels(list(Y_POS.keys()), fontsize=11)
    ax.set_ylim(-0.5, 4.5); ax.set_xlim(0, hours)
    ax.xaxis.set_major_locator(MultipleLocator(0.5))
    ax.grid(True, lw=0.7); ax.set_axisbelow(True)
    ax.set_ylabel("Sleep stage")
    ax.set_xlabel("Time since recording start (hours)")

def draw_selected_hypnogram(ax, t, y_step, stages, eps, marker_size, show_ticks=True):
    ax.step(t, y_step, where="post", color=LINE_LIGHT, lw=1.3, zorder=1)
    for s, ep in zip(stages, eps):
        if len(ep) == 0:
            continue
        x0 = ep.min() * 30 / 3600
        x1 = (ep.max() + 1) * 30 / 3600
        yy = Y_POS[s]
        ax.hlines(yy, x0, x1, color=COLORS[s], lw=9, zorder=2)
        if show_ticks:
            ax.vlines([x0, x1], yy - 0.3, yy + 0.3, color=LINE_DARK, lw=0.8, zorder=3)
        ax.scatter((x0 + x1) / 2, yy, s=marker_size, color=COLORS[s],
                   edgecolor=LINE_DARK, linewidth=0.6, zorder=3)

def plot_epoch_selection(labels, subject, max_rows=None):
    n = len(labels)
    hours = n * 30 / 3600

    old_split, old_stages, old_eps = load_selected(OLD_DIR, subject)
    new_split, new_stages, new_eps = load_selected(NEW_DIR, subject)
    old_found, new_found = old_split is not None, new_split is not None
    split = new_split or old_split
    split_txt = f" [{split}]" if split else ""

    print_selected(f"{subject} OLD", old_stages, old_eps, max_rows)
    print_selected(f"{subject} NEW", new_stages, new_eps, max_rows)

    def to_sel(stages, eps):
        sel = np.zeros(n, dtype=int)
        for s, ep in zip(stages, eps):
            ep = ep[(ep >= 0) & (ep < n)]
            sel[ep] = STAGES.index(s) + 1
        return sel
    old_sel, new_sel = to_sel(old_stages, old_eps), to_sel(new_stages, new_eps)

    exp_old = np.zeros(n, dtype=int)
    exp_new = np.zeros(n, dtype=int)
    for k, s in enumerate(STAGES, start=1):
        first, length = find_run(labels, s)
        if first is not None:
            exp_old[first:first + length] = k
        for st in find_all_runs(labels, s):
            exp_new[st:st + N_CONSEC] = k
    new_missing = [s for s in STAGES if len(find_all_runs(labels, s)) == 0]
    if new_missing:
        exp_new[:] = 0
    old_txt = "no file" if not old_found else ("match" if np.array_equal(old_sel, exp_old) else "MISMATCH")
    new_txt = "match" if np.array_equal(new_sel, exp_new) else "MISMATCH"
    if not new_found:
        new_txt = "no file (excluded)" if new_missing else "no file (UNEXPECTED)"
    print(f"\n{subject}: saved .npz vs recomputed from labels -> OLD: {old_txt} | NEW: {new_txt}")

    avail = {s: sum(l == s for l in labels) for s in STAGES}
    kept_old = {s: int((old_sel == k).sum()) for k, s in enumerate(STAGES, 1)}
    kept_new = {s: int((new_sel == k).sum()) for k, s in enumerate(STAGES, 1)}
    n_scored = sum(avail.values())
    n_old, n_new = sum(kept_old.values()), sum(kept_new.values())

    def sel_title(name, kept, n_sets, found, kind):
        if not found:
            why = f" (no 5-epoch run of {', '.join(new_missing)})" if (name == "NEW" and new_missing) else ""
            return f"{name} preprocessing: no .npz file, subject excluded{why}"
        return f"{name} {kind}: {n_sets} sets, {kept} epochs kept ({kept / n * 100:.1f}% of the recording)"

    fig = plt.figure(figsize=(16, 14))
    gs = fig.add_gridspec(4, 2, height_ratios=[2.2, 2.0, 2.0, 3.0], hspace=0.7, wspace=0.25,
                          left=0.06, right=0.90, top=0.94, bottom=0.05)

    t = np.arange(n + 1) * 30 / 3600
    y = np.array([Y_POS.get(l, np.nan) for l in labels] + [np.nan], dtype=float)
    y_step = np.append(y[:-1], y[-2])

    ax0 = fig.add_subplot(gs[0, :])
    ax0.step(t, y_step, where="post", color=LINE_DARK, lw=1.2)
    style_hyp_axis(ax0, hours)
    ax0.set_title(f"{subject}{split_txt}: whole-night hypnogram ({n} epochs of 30 s = {hours:.1f} h; "
                  f"{n_scored} scored as W/N1/N2/N3/R)", loc="left", fontsize=12)

    ax_old = fig.add_subplot(gs[1, :], sharex=ax0)
    draw_selected_hypnogram(ax_old, t, y_step, old_stages, old_eps, marker_size=110)
    style_hyp_axis(ax_old, hours)
    ax_old.set_title(sel_title("OLD", n_old, len(old_stages), old_found, "selected epochs on hypnogram"),
                     loc="left", fontsize=12)

    ax_new = fig.add_subplot(gs[2, :], sharex=ax0)
    draw_selected_hypnogram(ax_new, t, y_step, new_stages, new_eps, marker_size=30)
    style_hyp_axis(ax_new, hours)
    ax_new.set_title(sel_title("NEW", n_new, len(new_stages), new_found, "selected epochs on hypnogram"),
                     loc="left", fontsize=12)

    fig.legend(handles=[Line2D([0], [0], color=LINE_LIGHT, lw=2, label="whole night")]
                       + [Patch(facecolor=COLORS[s], edgecolor=LINE_LIGHT, label=s) for s in STAGES],
               loc="center left", bbox_to_anchor=(0.91, 0.68), frameon=False, title="kept epoch")

    ax3 = fig.add_subplot(gs[3, 0])
    x = np.arange(len(STAGES)); w = 0.27
    series = [("Available in night", [avail[s] for s in STAGES], C_AVAIL),
              ("Kept (OLD)", [kept_old[s] for s in STAGES], C_OLD),
              ("Kept (NEW)", [kept_new[s] for s in STAGES], C_NEW)]
    for j, (lab, vals, col) in enumerate(series):
        bars = ax3.bar(x + (j - 1) * w, vals, w, color=col, label=lab)
        for b_, v in zip(bars, vals):
            ax3.text(b_.get_x() + b_.get_width() / 2, v + max(max(avail.values()), 1) * 0.01,
                     str(v), ha="center", fontsize=8)
    ax3.set_xticks(x); ax3.set_xticklabels(["W", "N1", "N2", "N3", "REM"])
    ax3.set_ylabel("epochs"); ax3.legend(frameon=False)
    ax3.set_title("Epochs per stage: available vs kept", loc="left", fontsize=11)
    ax3.spines[["top", "right"]].set_visible(False)

    ax4 = fig.add_subplot(gs[3, 1])
    names = ["All epochs\nin recording", "Scored\n(W/N1/N2/N3/R)", "Kept\nNEW", "Kept\nOLD"]
    vals = [n, n_scored, n_new, n_old]
    cols = ["#C9C9D6", C_AVAIL, C_NEW, C_OLD]
    bars = ax4.barh(names[::-1], vals[::-1], color=cols[::-1])
    for b_, v in zip(bars, vals[::-1]):
        ax4.text(v + n * 0.01, b_.get_y() + b_.get_height() / 2,
                 f"{v}  ({v / n * 100:.1f}%)", va="center", fontsize=10)
    ax4.set_xlim(0, n * 1.25); ax4.set_xlabel("epochs")
    ax4.set_title("Total epochs: before vs after preprocessing", loc="left", fontsize=11)
    ax4.spines[["top", "right"]].set_visible(False)

    fig.suptitle(f"How many epochs does preprocessing keep? ({subject}{split_txt})", fontsize=14)
    save_fig(fig, f"chart1_epoch_selection_{subject}")
    return fig

for subj in SUBJECTS:
    try:
        labels = load_labels(f"{PROJECT_DIR}/dataset/HMC", subj)
    except Exception as e:
        print(f"ข้าม {subj}: {e}")
        continue
    fig = plot_epoch_selection(labels, subj, max_rows=20)
    plt.show()

## Section 3 จำนวนชุดข้อมูลต่อ stage (จากผลโน้ตบุ๊ก preprocess)



In [ ]:

STAGE_LABELS = ["W", "N1", "N2", "N3", "REM"]
SPLITS = ["train", "val", "test"]
SPLIT_COLORS = {"train": "#A7C7E7", "val": "#B8E0D2", "test": "#FFE3A3"}

old = {"train": [96, 96, 96, 96, 95],
       "val":   [24, 24, 24, 24, 24],
       "test":  [31, 31, 31, 29, 31]}
new = {"train": [1794, 351, 4134, 2409, 1825],
       "val":   [424, 76, 1031, 813, 534],
       "test":  [542, 158, 1156, 634, 542]}

def totals(d):
    return np.sum([d[s] for s in SPLITS], axis=0)

old_t, new_t = totals(old), totals(new)
assert old_t.sum() == 752 and new_t.sum() == 16423

def stacked(ax, d, title, subtitle):
    bottom = np.zeros(len(STAGE_LABELS))
    x = np.arange(len(STAGE_LABELS))
    for s in SPLITS:
        ax.bar(x, d[s], bottom=bottom, color=SPLIT_COLORS[s], label=s.capitalize(),
               width=0.62, edgecolor="white", linewidth=0.8)
        bottom += np.array(d[s])
    tot = bottom
    for xi, t in zip(x, tot):
        ax.text(xi, t + tot.max() * 0.015, f"{int(t):,}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(STAGE_LABELS, fontsize=11)
    ax.set_ylabel("Number of sets (sequences)")
    ax.set_ylim(0, tot.max() * 1.12)
    ax.set_title(title + "\n" + subtitle, fontsize=11, loc="left")
    ax.legend(title="Split", frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    ax.spines[["top", "right"]].set_visible(False)

## Section 4 กราฟที่ 2: แบบเก่า (02_preprocess_hmc)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
stacked(ax, old, "Old preprocessing (02_preprocess_hmc): sets per sleep stage",
        "first run per stage per subject | 752 sets, 3,719 epochs, 151 subjects")
fig.tight_layout()
save_fig(fig, "chart2_old_sets_per_stage")

## Section 5 กราฟที่ 3: แบบใหม่ (03_Preprocess_multi5)

ผลที่ได้:

- เหลือ **123 คน จาก 151 คน** (Train 79, Validation 21, Test 23) ตัดออก 28 คน (Train 17, Validation 3, Test 8)
- สาเหตุที่ตัดออก: ขาด N1 จำนวน 23 คน (รวม SN062 ซึ่งขาด W ด้วย), ขาด N3 จำนวน 3 คน (SN032, SN033, SN081) และขาด REM จำนวน 2 คน (SN112, SN057)
- ได้ข้อมูลรวม **16,423 กลุ่ม (82,115 epoch)** ทุกกลุ่มมีครบ 5 epoch


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
stacked(ax, new, "New preprocessing (03_Preprocess_multi5): sets per sleep stage",
        "all non-overlapping 5-epoch runs | 16,423 sets, 82,115 epochs, 123 subjects")
fig.tight_layout()
save_fig(fig, "chart3_new_sets_per_stage")

## Section 6 กราฟที่ 4: เปรียบเทียบแบบเก่ากับแบบใหม่

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={"width_ratios": [1.15, 1]})
x = np.arange(len(STAGE_LABELS)); w = 0.38
top = max(old_t.max(), new_t.max())
a.bar(x - w/2, old_t, w, color=C_OLD, label="Old (752 sets)")
a.bar(x + w/2, new_t, w, color=C_NEW, label="New (16,423 sets)")
for xi, v in zip(x - w/2, old_t): a.text(xi, v + top * 0.01, f"{v:,}", ha="center", fontsize=10)
for xi, v in zip(x + w/2, new_t): a.text(xi, v + top * 0.01, f"{v:,}", ha="center", fontsize=10)
a.set_ylim(0, top * 1.12)
a.set_xticks(x); a.set_xticklabels(STAGE_LABELS, fontsize=11)
a.set_ylabel("Number of sets")
a.set_title("A) Number of sets per stage", loc="left", fontsize=12)
a.legend(frameon=False, loc="upper left"); a.spines[["top", "right"]].set_visible(False)

po, pn = old_t / old_t.sum() * 100, new_t / new_t.sum() * 100
b.bar(x - w/2, po, w, color=C_OLD, label="Old")
b.bar(x + w/2, pn, w, color=C_NEW, label="New")
for xi, v in zip(x - w/2, po): b.text(xi, v + 0.6, f"{v:.1f}%", ha="center", fontsize=10)
for xi, v in zip(x + w/2, pn): b.text(xi, v + 0.6, f"{v:.1f}%", ha="center", fontsize=10)
b.set_xticks(x); b.set_xticklabels(STAGE_LABELS, fontsize=11)
b.set_ylabel("Share of all sets (%)"); b.set_ylim(0, max(pn.max(), po.max()) * 1.18)
b.set_title("B) Class proportion (share of total)", loc="left", fontsize=12)
b.legend(frameon=False); b.spines[["top", "right"]].set_visible(False)
fig.suptitle("Old vs New preprocessing: sets per sleep stage (train + val + test)", fontsize=13, y=1.0)
fig.tight_layout()
save_fig(fig, "chart4_old_vs_new")

print("old %:", np.round(po, 1), "new %:", np.round(pn, 1), "ratio new/old:", np.round(new_t / old_t, 1))

#Section 7 กราฟที่ 5: โครงสร้างชุดข้อมูล Multi5 ของผู้ถูกทดสอบ 1 คน

In [ ]:

from matplotlib.patches import Rectangle

SUBJECT_TO_PLOT = "SN001"

def plot_multi5_subject(subject):
    file_path = None
    split = None

    for sp in ["train", "val", "test"]:
        f = Path(NEW_DIR) / sp / f"{subject}.npz"
        if f.exists():
            file_path = f
            split = sp
            break

    if file_path is None:
        print(f"ไม่พบข้อมูล {subject}")
        return

    with np.load(file_path, allow_pickle=True) as d:
        stages = [str(s) for s in d["stage"]]
        epoch_indices = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]

    grouped = {
        s: sorted(
            [e for e, stage in zip(epoch_indices, stages) if stage == s],
            key=lambda e: e[0]
        )
        for s in STAGES
    }

    total_sets = len(stages)
    total_epochs = total_sets * N_CONSEC

    fig, ax = plt.subplots(
        figsize=(11, max(6, total_sets * 0.38 + 2.5))
    )

    y_labels = []
    y_positions = []
    current_y = 0

    for stage in STAGES:
        for i, epochs in enumerate(grouped[stage], start=1):
            for j, epoch in enumerate(epochs):
                ax.add_patch(Rectangle(
                    (j, current_y - 0.35), 1, 0.7,
                    facecolor=COLORS[stage],
                    edgecolor="white",
                    linewidth=1.5
                ))

                ax.text(
                    j + 0.5, current_y, str(epoch + 1),
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=LINE_DARK
                )

            y_labels.append(f"{stage} | Set {i}")
            y_positions.append(current_y)
            current_y += 1

        current_y += 0.5

    ax.set_xlim(0, 5)
    ax.set_ylim(current_y - 0.5, -0.5)

    ax.set_xticks(np.arange(5) + 0.5)
    ax.set_xticklabels([f"Epoch {i}" for i in range(1, 6)])

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_xlabel("Epoch within each set")
    ax.set_ylabel("Sleep stage and set")

    summary = " | ".join(
        f"{s}: {len(grouped[s])}" for s in STAGES
    )

    fig.suptitle(
        f"Multi5 Data Structure: {subject} ({split.upper()})\n"
        f"{total_sets} Sets | {total_epochs} Epochs",
        fontsize=13, fontweight="bold", x=0.08, ha="left"
    )

    ax.legend(
        handles=[Patch(facecolor=COLORS[s], label=s) for s in STAGES],
        title="Sleep stage",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False
    )

    ax.set_xticks(np.arange(6), minor=True)
    ax.grid(axis="x", which="minor", color="white", linewidth=2)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)

    fig.text(
        0.08, 0.015,
        f"{summary} sets",
        fontsize=10
    )

    fig.tight_layout(rect=[0.08, 0.05, 0.84, 0.91])

    save_fig(fig, f"chart5_multi5_subject_{subject}")
    plt.show()

    print(f"Subject: {subject} ({split})")
    print(summary)
    print(f"Total: {total_sets} sets, {total_epochs} epochs")


plot_multi5_subject(SUBJECT_TO_PLOT)

In [ ]:
from matplotlib.patches import Rectangle, Patch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SUBJECT_TO_PLOT = "SN001"
N_CONSEC = 5

def plot_multi5_subject(subject):
    file_path = None
    split = None

    for sp in ["train", "val", "test"]:
        f = Path(NEW_DIR) / sp / f"{subject}.npz"
        if f.exists():
            file_path = f
            split = sp
            break

    if file_path is None:
        print(f"ไม่พบข้อมูล {subject}")
        return

    with np.load(file_path, allow_pickle=True) as d:
        stages = [str(s) for s in d["stage"]]
        epoch_indices = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]

    grouped = {
        s: sorted(
            [e for e, stage in zip(epoch_indices, stages) if stage == s],
            key=lambda e: e[0]
        )
        for s in STAGES
    }

    total_sets = len(stages)
    total_epochs = total_sets * N_CONSEC

    fig_height = max(6, total_sets * 0.4 + 2.5)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    y_labels = []
    y_positions = []
    current_y = 0

    for stage in STAGES:
        if not grouped[stage]:
            continue
        for i, epochs in enumerate(grouped[stage], start=1):
            for j, epoch in enumerate(epochs):
                ax.add_patch(Rectangle(
                    (j, current_y - 0.35), 1, 0.7,
                    facecolor=COLORS[stage],
                    edgecolor="white",
                    linewidth=1.5,
                    antialiased=True
                ))

                ax.text(
                    j + 0.5, current_y, str(epoch + 1),
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=LINE_DARK
                )

            y_labels.append(f"{stage} | Set {i}")
            y_positions.append(current_y)
            current_y += 1

        current_y += 0.4

    ax.set_xlim(0, N_CONSEC)
    ax.set_ylim(current_y - 0.2, -0.5)

    ax.set_xticks(np.arange(N_CONSEC) + 0.5)
    ax.set_xticklabels([f"Epoch {i+1}" for i in range(N_CONSEC)], fontsize=9)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_xlabel("Epoch within each set", labelpad=10, fontsize=10, fontweight="bold")
    ax.set_ylabel("Sleep stage and set", labelpad=10, fontsize=10, fontweight="bold")

    summary = " | ".join(
        f"{s}: {len(grouped[s])}" for s in STAGES
    )

    fig.suptitle(
        f"Multi5 Data Structure: {subject} ({split.upper()})\n"
        f"{total_sets} Sets | {total_epochs} Epochs",
        fontsize=13, fontweight="bold", x=0.07, y=0.96, ha="left"
    )

    ax.legend(
        handles=[Patch(facecolor=COLORS[s], label=s) for s in STAGES],
        title="Sleep stage",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False
    )

    ax.set_xticks(np.arange(N_CONSEC + 1), minor=True)
    ax.grid(axis="x", which="minor", color="white", linewidth=2, alpha=0.8)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#cccccc")

    fig.text(
        0.07, 0.02,
        f"Summary: {summary} sets",
        fontsize=9, color="#555555"
    )

    fig.tight_layout(rect=[0.07, 0.06, 0.83, 0.90])

    save_fig(fig, f"chart5_multi5_subject_{subject}")
    plt.show()

    print(f"Subject: {subject} ({split})")
    print(summary)
    print(f"Total: {total_sets} sets, {total_epochs} epochs")

plot_multi5_subject(SUBJECT_TO_PLOT)

In [ ]:
from matplotlib.patches import Rectangle, Patch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SUBJECT_TO_PLOT = "SN004"
N_CONSEC = 5

def plot_multi5_subject(subject):
    file_path = None
    split = None

    for sp in ["train", "val", "test"]:
        f = Path(NEW_DIR) / sp / f"{subject}.npz"
        if f.exists():
            file_path = f
            split = sp
            break

    if file_path is None:
        print(f"ไม่พบข้อมูล {subject}")
        return

    with np.load(file_path, allow_pickle=True) as d:
        stages = [str(s) for s in d["stage"]]
        epoch_indices = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]

    grouped = {
        s: sorted(
            [e for e, stage in zip(epoch_indices, stages) if stage == s],
            key=lambda e: e[0]
        )
        for s in STAGES
    }

    total_sets = len(stages)
    total_epochs = total_sets * N_CONSEC

    fig_height = max(6, total_sets * 0.4 + 2.5)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    y_labels = []
    y_positions = []
    current_y = 0

    for stage in STAGES:
        if not grouped[stage]:
            continue
        for i, epochs in enumerate(grouped[stage], start=1):
            for j, epoch in enumerate(epochs):
                ax.add_patch(Rectangle(
                    (j, current_y - 0.35), 1, 0.7,
                    facecolor=COLORS[stage],
                    edgecolor="white",
                    linewidth=1.5,
                    antialiased=True
                ))

                ax.text(
                    j + 0.5, current_y, str(epoch + 1),
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=LINE_DARK
                )

            y_labels.append(f"{stage} | Set {i}")
            y_positions.append(current_y)
            current_y += 1

        current_y += 0.4

    ax.set_xlim(0, N_CONSEC)
    ax.set_ylim(current_y - 0.2, -0.5)

    ax.set_xticks(np.arange(N_CONSEC) + 0.5)
    ax.set_xticklabels([f"Epoch {i+1}" for i in range(N_CONSEC)], fontsize=9)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_xlabel("Epoch within each set", labelpad=10, fontsize=10, fontweight="bold")
    ax.set_ylabel("Sleep stage and set", labelpad=10, fontsize=10, fontweight="bold")

    summary = " | ".join(
        f"{s}: {len(grouped[s])}" for s in STAGES
    )

    fig.suptitle(
        f"Multi5 Data Structure: {subject} ({split.upper()})\n"
        f"{total_sets} Sets | {total_epochs} Epochs",
        fontsize=13, fontweight="bold", x=0.07, y=0.96, ha="left"
    )

    ax.legend(
        handles=[Patch(facecolor=COLORS[s], label=s) for s in STAGES],
        title="Sleep stage",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False
    )

    ax.set_xticks(np.arange(N_CONSEC + 1), minor=True)
    ax.grid(axis="x", which="minor", color="white", linewidth=2, alpha=0.8)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#cccccc")

    fig.text(
        0.07, 0.02,
        f"Summary: {summary} sets",
        fontsize=9, color="#555555"
    )

    fig.tight_layout(rect=[0.07, 0.06, 0.83, 0.90])

    save_fig(fig, f"chart5_multi5_subject_{subject}")
    plt.show()

    print(f"Subject: {subject} ({split})")
    print(summary)
    print(f"Total: {total_sets} sets, {total_epochs} epochs")

plot_multi5_subject(SUBJECT_TO_PLOT)

In [ ]:
from matplotlib.patches import Rectangle, Patch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SUBJECT_TO_PLOT = "SN006"
N_CONSEC = 5

def plot_multi5_subject(subject):
    file_path = None
    split = None

    for sp in ["train", "val", "test"]:
        f = Path(NEW_DIR) / sp / f"{subject}.npz"
        if f.exists():
            file_path = f
            split = sp
            break

    if file_path is None:
        print(f"ไม่พบข้อมูล {subject}")
        return

    with np.load(file_path, allow_pickle=True) as d:
        stages = [str(s) for s in d["stage"]]
        epoch_indices = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]

    grouped = {
        s: sorted(
            [e for e, stage in zip(epoch_indices, stages) if stage == s],
            key=lambda e: e[0]
        )
        for s in STAGES
    }

    total_sets = len(stages)
    total_epochs = total_sets * N_CONSEC

    fig_height = max(6, total_sets * 0.4 + 2.5)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    y_labels = []
    y_positions = []
    current_y = 0

    for stage in STAGES:
        if not grouped[stage]:
            continue
        for i, epochs in enumerate(grouped[stage], start=1):
            for j, epoch in enumerate(epochs):
                ax.add_patch(Rectangle(
                    (j, current_y - 0.35), 1, 0.7,
                    facecolor=COLORS[stage],
                    edgecolor="white",
                    linewidth=1.5,
                    antialiased=True
                ))

                ax.text(
                    j + 0.5, current_y, str(epoch + 1),
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=LINE_DARK
                )

            y_labels.append(f"{stage} | Set {i}")
            y_positions.append(current_y)
            current_y += 1

        current_y += 0.4

    ax.set_xlim(0, N_CONSEC)
    ax.set_ylim(current_y - 0.2, -0.5)

    ax.set_xticks(np.arange(N_CONSEC) + 0.5)
    ax.set_xticklabels([f"Epoch {i+1}" for i in range(N_CONSEC)], fontsize=9)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_xlabel("Epoch within each set", labelpad=10, fontsize=10, fontweight="bold")
    ax.set_ylabel("Sleep stage and set", labelpad=10, fontsize=10, fontweight="bold")

    summary = " | ".join(
        f"{s}: {len(grouped[s])}" for s in STAGES
    )

    fig.suptitle(
        f"Multi5 Data Structure: {subject} ({split.upper()})\n"
        f"{total_sets} Sets | {total_epochs} Epochs",
        fontsize=13, fontweight="bold", x=0.07, y=0.96, ha="left"
    )

    ax.legend(
        handles=[Patch(facecolor=COLORS[s], label=s) for s in STAGES],
        title="Sleep stage",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False
    )

    ax.set_xticks(np.arange(N_CONSEC + 1), minor=True)
    ax.grid(axis="x", which="minor", color="white", linewidth=2, alpha=0.8)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#cccccc")

    fig.text(
        0.07, 0.02,
        f"Summary: {summary} sets",
        fontsize=9, color="#555555"
    )

    fig.tight_layout(rect=[0.07, 0.06, 0.83, 0.90])

    save_fig(fig, f"chart5_multi5_subject_{subject}")
    plt.show()

    print(f"Subject: {subject} ({split})")
    print(summary)
    print(f"Total: {total_sets} sets, {total_epochs} epochs")

plot_multi5_subject(SUBJECT_TO_PLOT)

In [ ]:
from matplotlib.patches import Rectangle, Patch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SUBJECT_TO_PLOT = "SN002"
N_CONSEC = 5

def plot_multi5_subject(subject):
    file_path = None
    split = None

    for sp in ["train", "val", "test"]:
        f = Path(NEW_DIR) / sp / f"{subject}.npz"
        if f.exists():
            file_path = f
            split = sp
            break

    if file_path is None:
        print(f"ไม่พบข้อมูล {subject}")
        return

    with np.load(file_path, allow_pickle=True) as d:
        stages = [str(s) for s in d["stage"]]
        epoch_indices = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]

    grouped = {
        s: sorted(
            [e for e, stage in zip(epoch_indices, stages) if stage == s],
            key=lambda e: e[0]
        )
        for s in STAGES
    }

    total_sets = len(stages)
    total_epochs = total_sets * N_CONSEC

    fig_height = max(6, total_sets * 0.4 + 2.5)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    y_labels = []
    y_positions = []
    current_y = 0

    for stage in STAGES:
        if not grouped[stage]:
            continue
        for i, epochs in enumerate(grouped[stage], start=1):
            for j, epoch in enumerate(epochs):
                ax.add_patch(Rectangle(
                    (j, current_y - 0.35), 1, 0.7,
                    facecolor=COLORS[stage],
                    edgecolor="white",
                    linewidth=1.5,
                    antialiased=True
                ))

                ax.text(
                    j + 0.5, current_y, str(epoch + 1),
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=LINE_DARK
                )

            y_labels.append(f"{stage} | Set {i}")
            y_positions.append(current_y)
            current_y += 1

        current_y += 0.4

    ax.set_xlim(0, N_CONSEC)
    ax.set_ylim(current_y - 0.2, -0.5)

    ax.set_xticks(np.arange(N_CONSEC) + 0.5)
    ax.set_xticklabels([f"Epoch {i+1}" for i in range(N_CONSEC)], fontsize=9)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_xlabel("Epoch within each set", labelpad=10, fontsize=10, fontweight="bold")
    ax.set_ylabel("Sleep stage and set", labelpad=10, fontsize=10, fontweight="bold")

    summary = " | ".join(
        f"{s}: {len(grouped[s])}" for s in STAGES
    )

    fig.suptitle(
        f"Multi5 Data Structure: {subject} ({split.upper()})\n"
        f"{total_sets} Sets | {total_epochs} Epochs",
        fontsize=13, fontweight="bold", x=0.07, y=0.96, ha="left"
    )

    ax.legend(
        handles=[Patch(facecolor=COLORS[s], label=s) for s in STAGES],
        title="Sleep stage",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False
    )

    ax.set_xticks(np.arange(N_CONSEC + 1), minor=True)
    ax.grid(axis="x", which="minor", color="white", linewidth=2, alpha=0.8)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#cccccc")

    fig.text(
        0.07, 0.02,
        f"Summary: {summary} sets",
        fontsize=9, color="#555555"
    )

    fig.tight_layout(rect=[0.07, 0.06, 0.83, 0.90])

    save_fig(fig, f"chart5_multi5_subject_{subject}")
    plt.show()

    print(f"Subject: {subject} ({split})")
    print(summary)
    print(f"Total: {total_sets} sets, {total_epochs} epochs")

plot_multi5_subject(SUBJECT_TO_PLOT)

In [ ]:
from matplotlib.patches import Rectangle, Patch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SUBJECT_TO_PLOT = "SN010"
N_CONSEC = 5

def plot_multi5_subject(subject):
    file_path = None
    split = None

    for sp in ["train", "val", "test"]:
        f = Path(NEW_DIR) / sp / f"{subject}.npz"
        if f.exists():
            file_path = f
            split = sp
            break

    if file_path is None:
        print(f"ไม่พบข้อมูล {subject}")
        return

    with np.load(file_path, allow_pickle=True) as d:
        stages = [str(s) for s in d["stage"]]
        epoch_indices = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]

    grouped = {
        s: sorted(
            [e for e, stage in zip(epoch_indices, stages) if stage == s],
            key=lambda e: e[0]
        )
        for s in STAGES
    }

    total_sets = len(stages)
    total_epochs = total_sets * N_CONSEC

    fig_height = max(6, total_sets * 0.4 + 2.5)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    y_labels = []
    y_positions = []
    current_y = 0

    for stage in STAGES:
        if not grouped[stage]:
            continue
        for i, epochs in enumerate(grouped[stage], start=1):
            for j, epoch in enumerate(epochs):
                ax.add_patch(Rectangle(
                    (j, current_y - 0.35), 1, 0.7,
                    facecolor=COLORS[stage],
                    edgecolor="white",
                    linewidth=1.5,
                    antialiased=True
                ))

                ax.text(
                    j + 0.5, current_y, str(epoch + 1),
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=LINE_DARK
                )

            y_labels.append(f"{stage} | Set {i}")
            y_positions.append(current_y)
            current_y += 1

        current_y += 0.4

    ax.set_xlim(0, N_CONSEC)
    ax.set_ylim(current_y - 0.2, -0.5)

    ax.set_xticks(np.arange(N_CONSEC) + 0.5)
    ax.set_xticklabels([f"Epoch {i+1}" for i in range(N_CONSEC)], fontsize=9)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_xlabel("Epoch within each set", labelpad=10, fontsize=10, fontweight="bold")
    ax.set_ylabel("Sleep stage and set", labelpad=10, fontsize=10, fontweight="bold")

    summary = " | ".join(
        f"{s}: {len(grouped[s])}" for s in STAGES
    )

    fig.suptitle(
        f"Multi5 Data Structure: {subject} ({split.upper()})\n"
        f"{total_sets} Sets | {total_epochs} Epochs",
        fontsize=13, fontweight="bold", x=0.07, y=0.96, ha="left"
    )

    ax.legend(
        handles=[Patch(facecolor=COLORS[s], label=s) for s in STAGES],
        title="Sleep stage",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False
    )

    ax.set_xticks(np.arange(N_CONSEC + 1), minor=True)
    ax.grid(axis="x", which="minor", color="white", linewidth=2, alpha=0.8)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#cccccc")

    fig.text(
        0.07, 0.02,
        f"Summary: {summary} sets",
        fontsize=9, color="#555555"
    )

    fig.tight_layout(rect=[0.07, 0.06, 0.83, 0.90])

    save_fig(fig, f"chart5_multi5_subject_{subject}")
    plt.show()

    print(f"Subject: {subject} ({split})")
    print(summary)
    print(f"Total: {total_sets} sets, {total_epochs} epochs")

plot_multi5_subject(SUBJECT_TO_PLOT)

In [ ]:

from matplotlib.patches import Rectangle

SUBJECT_TO_PLOT = "SN001"

def plot_multi5_subject(subject):
    file_path = None
    split = None

    for sp in ["train", "val", "test"]:
        f = Path(OLD_DIR) / sp / f"{subject}.npz"
        if f.exists():
            file_path = f
            split = sp
            break

    if file_path is None:
        print(f"ไม่พบข้อมูล {subject}")
        return

    with np.load(file_path, allow_pickle=True) as d:
        stages = [str(s) for s in d["stage"]]
        epoch_indices = [np.asarray(e, dtype=int) for e in d["epoch_indices"]]

    grouped = {
        s: sorted(
            [e for e, stage in zip(epoch_indices, stages) if stage == s],
            key=lambda e: e[0]
        )
        for s in STAGES
    }

    total_sets = len(stages)
    total_epochs = total_sets * N_CONSEC

    fig, ax = plt.subplots(
        figsize=(11, max(6, total_sets * 0.38 + 2.5))
    )

    y_labels = []
    y_positions = []
    current_y = 0

    for stage in STAGES:
        for i, epochs in enumerate(grouped[stage], start=1):
            for j, epoch in enumerate(epochs):
                ax.add_patch(Rectangle(
                    (j, current_y - 0.35), 1, 0.7,
                    facecolor=COLORS[stage],
                    edgecolor="white",
                    linewidth=1.5
                ))

                ax.text(
                    j + 0.5, current_y, str(epoch + 1),
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=LINE_DARK
                )

            y_labels.append(f"{stage} | Set {i}")
            y_positions.append(current_y)
            current_y += 1

        current_y += 0.5

    ax.set_xlim(0, 5)
    ax.set_ylim(current_y - 0.5, -0.5)

    ax.set_xticks(np.arange(5) + 0.5)
    ax.set_xticklabels([f"Epoch {i}" for i in range(1, 6)])

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_xlabel("Epoch within each set")
    ax.set_ylabel("Sleep stage and set")

    summary = " | ".join(
        f"{s}: {len(grouped[s])}" for s in STAGES
    )

    fig.suptitle(
        f"Multi5 Data Structure: {subject} ({split.upper()})\n"
        f"{total_sets} Sets | {total_epochs} Epochs",
        fontsize=13, fontweight="bold", x=0.08, ha="left"
    )

    ax.legend(
        handles=[Patch(facecolor=COLORS[s], label=s) for s in STAGES],
        title="Sleep stage",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False
    )

    ax.set_xticks(np.arange(6), minor=True)
    ax.grid(axis="x", which="minor", color="white", linewidth=2)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)

    fig.text(
        0.08, 0.015,
        f"{summary} sets",
        fontsize=10
    )

    fig.tight_layout(rect=[0.08, 0.05, 0.84, 0.91])

    save_fig(fig, f"chart5_multi5_subject_{subject}")
    plt.show()

    print(f"Subject: {subject} ({split})")
    print(summary)
    print(f"Total: {total_sets} sets, {total_epochs} epochs")


plot_multi5_subject(SUBJECT_TO_PLOT)